# ML Assignment 2 — Income Classification (UCI Adult Dataset)

This notebook implements and evaluates 5 classification models (Logistic Regression, Decision Tree, kNN, Naive Bayes, Random Forest) on the UCI Adult / Census Income dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)

RANDOM_STATE = 42

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../adult_income_raw.csv')
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()

## 2. Data Cleaning & Preprocessing

- Replace `?` with missing values
- Drop rows with a missing target
- Encode target: `<=50K` -> 0, `>50K` -> 1

In [ ]:
df = df.replace(r'^\s*\?\s*$', np.nan, regex=True)
df = df.dropna(subset=['income'])
df['income'] = df['income'].astype(str).str.strip().str.replace('.', '', regex=False)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
df = df.dropna(subset=['income'])
df['income'] = df['income'].astype(int)
df.isna().sum()

### Quick EDA

In [ ]:
df['income'].value_counts(normalize=True).rename({0: '<=50K', 1: '>50K'})

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sns.countplot(x=df['income'].map({0: '<=50K', 1: '>50K'}), ax=ax)
ax.set_title('Target class distribution')
plt.show()

## 3. Feature / Target Split and Preprocessing Pipeline

In [ ]:
NUMERIC_FEATURES = ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']
CATEGORICAL_FEATURES = ['workclass', 'education', 'marital.status', 'occupation',
                         'relationship', 'race', 'sex', 'native.country']
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[FEATURE_COLS]
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, NUMERIC_FEATURES),
    ('cat', categorical_pipeline, CATEGORICAL_FEATURES),
])

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)
X_train_t.shape

## 4. Train 5 Classification Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'kNN': KNeighborsClassifier(n_neighbors=15),
    'Naive Bayes': GaussianNB(),
    'Random Forest (Ensemble)': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
}

fitted_models = {}
for name, model in models.items():
    model.fit(X_train_t, y_train)
    fitted_models[name] = model
    print(f'Trained {name}')

## 5. Evaluate: Accuracy, AUC, Precision, Recall, F1, MCC

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_proba),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'MCC': matthews_corrcoef(y_test, y_pred),
    }

results = []
for name, model in fitted_models.items():
    metrics = evaluate(model, X_test_t, y_test)
    metrics['Model'] = name
    results.append(metrics)

metrics_df = pd.DataFrame(results)[['Model', 'Accuracy', 'AUC', 'Precision', 'Recall', 'F1', 'MCC']]
metrics_df

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, model) in zip(axes, fitted_models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test_t))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['<=50K', '>50K'], yticklabels=['<=50K', '>50K'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 7. Save Models, Preprocessor, and Metrics

(Also done by `model/train_models.py`, which this notebook mirrors.)

In [ ]:
import joblib
import os

joblib.dump(preprocessor, '../model/preprocessor.pkl')
file_map = {
    'Logistic Regression': 'logistic_regression_model.pkl',
    'Decision Tree': 'decision_tree_model.pkl',
    'kNN': 'knn_model.pkl',
    'Naive Bayes': 'naive_bayes_model.pkl',
    'Random Forest (Ensemble)': 'random_forest_model.pkl',
}
for name, model in fitted_models.items():
    joblib.dump(model, f'../model/{file_map[name]}')

metrics_df.to_csv('../model/metrics.csv', index=False)
print('Saved all models, preprocessor, and metrics.')

## 8. Final Comparison Table

In [ ]:
metrics_df.round(4)